# Sentiment Analysis with VADER to BigQuery

## Purpose
The VADER (Valence Aware Dictionary and sEntiment Reasoner) sentiment analysis model is a lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media. Since VADER cannot be executed directly within standard BigQuery SQL, this standalone Python notebook is used to:
1. Fetch raw review data from BigQuery.
2. Perform sentiment scoring using the VADER library.
3. Write the resulting sentiment scores and risk flags back to BigQuery.

These results serve as input for the `intermediate` layer in the dbt project, enabling downstream business logic and modeling.

## Step 1: Configuration and Setup
Initialize the BigQuery client and define the source/target tables.

In [6]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from google.cloud import bigquery
from datetime import datetime
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Configuration
PROJECT_ID = os.getenv("PROJECT_ID")
DATASET_ID = os.getenv("DATASET_ID")
OUTPUT_DATASET = "dbt_ecommerce_intermediate"

SOURCE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.raw_reviews"
DETAIL_TABLE = f"{PROJECT_ID}.{OUTPUT_DATASET}.int_review_sentiment_detail"
USER_TABLE = f"{PROJECT_ID}.{OUTPUT_DATASET}.int_review_sentiment_user"

client = bigquery.Client(project=PROJECT_ID)
print(f"[{datetime.now()}] Configured for project: {PROJECT_ID}")

[2026-06-08 12:28:27.532714] Configured for project: my-project-for-bigquery-445809


## Step 2: NLTK Initialization
Ensure the VADER lexicon is downloaded and initialize the analyzer.

In [7]:
try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    print("Downloading VADER lexicon...")
    nltk.download("vader_lexicon", quiet=True)

sid = SentimentIntensityAnalyzer()
print("Sentiment Intensity Analyzer initialized.")

Sentiment Intensity Analyzer initialized.


## Step 3: Load Data from BigQuery
Fetch review data including `review_id`, `user_id`, and the review text itself.

In [8]:
print(f"[{datetime.now()}] Reading data from {SOURCE_TABLE}...")
query = f"SELECT review_id, user_id, product_id, rating, review_text FROM `{SOURCE_TABLE}`"
reviews = client.query(query).to_dataframe()
print(f"Loaded {len(reviews)} reviews.")

[2026-06-08 12:28:35.687817] Reading data from my-project-for-bigquery-445809.raw_ecommerce.raw_reviews...


/Volumes/Transcend/Profile/marketing/ecommerce_dataset/venv/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 15000 reviews.


## Step 4: Perform Sentiment Analysis
Calculate the `compound_score` and determine `risk_flag` based on sentiment and rating.

In [9]:
print(f"[{datetime.now()}] Performing sentiment analysis...")

# Calculate sentiment scores
reviews["compound_score"] = reviews["review_text"].apply(
    lambda x: sid.polarity_scores(str(x))["compound"]
)

# Risk Flag: Compound Score < -0.05 and Rating <= 3
reviews["risk_flag"] = (
    (reviews["compound_score"] < -0.05) & (reviews["rating"] <= 3)
).astype(int)

print("Sentiment analysis complete.")
reviews.head()

[2026-06-08 12:28:42.281901] Performing sentiment analysis...
Sentiment analysis complete.


,review_id,user_id,product_id,rating,review_text,compound_score,risk_flag
0,R00009994,U003427,P000424,1,Color was different from images.,0.0,0
1,R00035560,U005160,P001370,1,Color was different from images.,0.0,0
2,R00025818,U003868,P001745,1,Color was different from images.,0.0,0
3,R00013011,U008256,P001126,1,Color was different from images.,0.0,0
4,R00036280,U005171,P001055,1,Color was different from images.,0.0,0


## Step 5: Export Results back to BigQuery
Prepare and load the detail and user-level aggregate tables.

In [10]:
# Prepare Detail Table
sentiment_detail = reviews[
    ["review_id", "user_id", "product_id", "compound_score", "risk_flag"]
]

# Prepare User Table (Aggregated)
sentiment_user = (
    reviews.groupby("user_id")
    .agg({"compound_score": "mean", "risk_flag": "max"})
    .reset_index()
)

print(f"[{datetime.now()}] Writing detail results to {DETAIL_TABLE}...")
client.load_table_from_dataframe(sentiment_detail, DETAIL_TABLE).result()

print(f"[{datetime.now()}] Writing user results to {USER_TABLE}...")
client.load_table_from_dataframe(sentiment_user, USER_TABLE).result()

print(f"[{datetime.now()}] All results exported successfully.")

[2026-06-08 12:28:45.656611] Writing detail results to my-project-for-bigquery-445809.dbt_ecommerce_intermediate.int_review_sentiment_detail...
[2026-06-08 12:28:49.563380] Writing user results to my-project-for-bigquery-445809.dbt_ecommerce_intermediate.int_review_sentiment_user...
[2026-06-08 12:28:53.091014] All results exported successfully.
